---
# Notebook 8 – Feature Scaling
## 1. Why feature scaling?

Features can have different units and ranges. For example, annual income may range from thousands to millions, while age may range from 18 to 90. Some algorithms use distances, dot products, or gradient-based optimization, so large-scale features can dominate.

**Scale-sensitive algorithms:** K-Nearest Neighbors (KNN), K-Means, Support Vector Machines (especially RBF kernel), PCA, and many gradient-based models such as logistic regression and neural networks.

**Usually scale-insensitive algorithms:** Decision trees, Random Forests, and many gradient-boosted tree models. Scaling is usually not required for their split decisions, although preprocessing requirements can vary by implementation and workflow.

## 2. Scaling techniques

- **Min-Max scaling:** maps values to a chosen range, commonly 0–1. Sensitive to outliers.
- **Standardization:** subtracts the training mean and divides by the training standard deviation. Produces approximately zero mean and unit variance; does not guarantee normality.
- **Robust scaling:** uses the median and interquartile range (IQR). Often more resistant to outliers.
- **MaxAbs scaling:** divides by the maximum absolute value; preserves zero and is useful for sparse data.
- **Normalization (row-wise):** scales each sample vector to unit norm. This is different from Min-Max scaling, which scales each feature.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler,
    Normalizer
)

rng = np.random.default_rng(42)
scaling_df = pd.DataFrame({
    "income": np.r_[rng.normal(50000, 12000, 99), 250000],
    "age": rng.integers(18, 70, 100)
})
scaling_df.head()

In [ ]:
scalers = {
    "Original": None,
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
    "RobustScaler": RobustScaler(),
    "MaxAbsScaler": MaxAbsScaler()
}

scaled_results = {}
for name, scaler in scalers.items():
    scaled_results[name] = (
        scaling_df.copy() if scaler is None
        else pd.DataFrame(scaler.fit_transform(scaling_df),
                          columns=scaling_df.columns)
    )

comparison = pd.concat(
    {name: frame.agg(["min", "median", "max", "mean", "std"]).round(3)
     for name, frame in scaled_results.items()},
    names=["Technique", "Statistic"]
)
comparison

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, (name, frame) in zip(axes, list(scaled_results.items())[1:]):
    ax.boxplot(frame["income"], vert=True)
    ax.set_title(name)
    ax.set_ylabel("Scaled income")
plt.tight_layout()
plt.show()

### Interpretation: outliers and scaling

The extreme income value affects the minimum and maximum. **MinMaxScaler** compresses most ordinary observations into a small part of the 0–1 range. **StandardScaler** is also affected because the outlier changes the mean and standard deviation. **RobustScaler** uses the median and IQR, so typical observations are less affected. **MaxAbsScaler** is sensitive to the largest absolute value.

No scaler automatically removes outliers. Investigate whether an extreme value is a data error or a valid rare event before deciding how to handle it.

| Technique | Main idea | Typical use / caution |
|---|---|---|
| MinMaxScaler | Maps each feature to a range | Useful when a bounded range is wanted; outlier-sensitive |
| StandardScaler | Mean 0, standard deviation 1 | Common for linear, distance-based, and gradient-based models |
| RobustScaler | Median 0, scales by IQR | Useful when outliers are present |
| MaxAbsScaler | Divides by max absolute value | Preserves zero; useful for sparse data |
| Normalizer | Scales each row/vector to unit norm | Often used for text/vector data; not feature-wise scaling |

**Important:** fit the scaler on training data only, then use that fitted scaler to transform validation/test data. This prevents information from the test set influencing preprocessing.

---
# Notebook 9 – Data Transformation
## 1. Why transform data?

Transformations can reduce right-skew, stabilize variance, make relationships easier for a model to learn, or reduce the influence of very large positive values. They do not guarantee improved model performance; compare results using an appropriate validation strategy.

- **Log transformation:** useful for positive, strongly right-skewed values; `log1p(x)` supports zero.
- **Square-root transformation:** often a gentler option for non-negative count-like values.
- **Power transformation:** a family of transformations that can help reduce skew.
- **Box-Cox:** requires strictly positive values.
- **Yeo-Johnson:** supports positive, zero, and negative values.
- **Normalization / standardization:** rescale values; they do not necessarily fix skewness.

In [ ]:
from sklearn.preprocessing import PowerTransformer, StandardScaler, MinMaxScaler

rng = np.random.default_rng(7)
transform_df = pd.DataFrame({
    "spending": rng.lognormal(mean=2.0, sigma=0.9, size=800)
})
transform_df["spending"].describe()

In [ ]:
x = transform_df["spending"]
transformations = {
    "Original": x,
    "Log1p": np.log1p(x),
    "Square root": np.sqrt(x),
    "Box-Cox": pd.Series(
        PowerTransformer(method="box-cox", standardize=False)
        .fit_transform(x.to_numpy().reshape(-1, 1)).ravel()
    ),
    "Yeo-Johnson": pd.Series(
        PowerTransformer(method="yeo-johnson", standardize=False)
        .fit_transform(x.to_numpy().reshape(-1, 1)).ravel()
    )
}

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, (name, values) in zip(axes.flat, transformations.items()):
    ax.hist(values, bins=30, edgecolor="black")
    ax.set_title(name)
    ax.set_ylabel("Frequency")
axes.flat[-1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
pd.DataFrame({
    name: {
        "mean": values.mean(),
        "std": values.std(),
        "skewness": values.skew(),
        "min": values.min(),
        "max": values.max()
    }
    for name, values in transformations.items()
}).round(3)

### Compare before and after

Inspect the histograms and skewness values. A right-skewed feature often has a long tail toward large values; log and power transforms may make the distribution more symmetric. Results depend on the data, so use the plots and statistics rather than assuming a transformation worked.

**Choosing a transformation**
- Use `np.log1p(x)` for non-negative data when a log-like transformation is appropriate.
- Use `np.sqrt(x)` for non-negative counts when a milder transformation is suitable.
- Use `PowerTransformer(method="box-cox")` only when all values are strictly positive.
- Use `PowerTransformer(method="yeo-johnson")` when values can include zero or negatives.
- Use `StandardScaler` or `MinMaxScaler` when the goal is scale, not skewness.

In predictive workflows, estimate transformation parameters on training data only. For repeatable workflows, place transformations inside a Scikit-learn pipeline.

---
# Notebook 10 – Feature Selection
## 1. What is feature selection?

Feature selection keeps a useful subset of the original features. It is different from feature extraction, which creates new features (for example, PCA components).

- **Relevant features:** contain information useful for predicting the target.
- **Irrelevant features:** add little or no predictive information.
- **Redundant features:** repeat information already captured by other features.

Unnecessary features can increase training time, add noise, increase overfitting risk, make explanations harder, and cause distance-based models to behave poorly. However, removing a feature based only on intuition or correlation can discard useful signal.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import (
    VarianceThreshold, SelectKBest, mutual_info_classif, RFE
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X_array, y = make_classification(
    n_samples=700, n_features=10, n_informative=4, n_redundant=3,
    n_repeated=0, n_classes=2, random_state=42, shuffle=True
)
feature_names = [f"feature_{i}" for i in range(X_array.shape[1])]
X = pd.DataFrame(X_array, columns=feature_names)
# Add one constant feature to demonstrate variance threshold
X["constant_feature"] = 1.0

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
X_train.shape, X_test.shape

In [ ]:
# Variance Threshold: removes features with variance at or below the threshold.
variance_selector = VarianceThreshold(threshold=0.0)
variance_selector.fit(X_train)
kept_by_variance = X_train.columns[variance_selector.get_support()].tolist()
removed_by_variance = X_train.columns[~variance_selector.get_support()].tolist()
print("Removed:", removed_by_variance)
print("Kept:", kept_by_variance)

In [ ]:
# Univariate feature selection using mutual information.
# Fit on training data only.
mi_selector = SelectKBest(score_func=mutual_info_classif, k=5)
mi_selector.fit(X_train[kept_by_variance], y_train)
mi_scores = pd.Series(
    mi_selector.scores_,
    index=kept_by_variance
).sort_values(ascending=False)
mi_scores

In [ ]:
# Recursive Feature Elimination (RFE) with logistic regression.
rfe_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
rfe = RFE(estimator=LogisticRegression(max_iter=2000), n_features_to_select=5)
# RFE's estimator is fit on the input feature scale; this synthetic dataset is suitable for demonstration.
rfe.fit(X_train[kept_by_variance], y_train)
print("RFE selected:", X_train[kept_by_variance].columns[rfe.support_].tolist())

In [ ]:
# Model-based feature importance (tree-based example).
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train[kept_by_variance], y_train)
importance = pd.Series(
    rf.feature_importances_, index=kept_by_variance
).sort_values(ascending=False)
importance.plot(kind="bar", figsize=(9, 3), title="Random Forest feature importance")
plt.ylabel("Importance")
plt.tight_layout()
plt.show()

### Feature-selection methods compared

| Method | How it works | Main caution |
|---|---|---|
| Correlation-based selection | Finds highly correlated features, often for redundancy review | Correlation with target may miss nonlinear signal; correlation between features is not proof one is useless |
| Variance Threshold | Removes constant / near-constant features | A low-variance feature can still be predictive |
| Univariate selection | Scores each feature individually, e.g. statistical tests | Can miss interactions between features |
| Mutual Information | Detects statistical dependence, including nonlinear relationships | Estimates can be noisy; fit on training data |
| RFE | Repeatedly removes less useful features based on an estimator | More computationally expensive; tune within validation/CV |
| Feature importance | Ranks features using a fitted model | Importance may be biased or unstable; correlated features can share importance |

**Good practice:** perform feature selection inside the training workflow (ideally a pipeline and cross-validation). Do not select features using the full dataset before the train/test split.

---
# Notebook 11 – Handling Imbalanced Data
## 1. What is class imbalance?

A classification dataset is imbalanced when one class has many more examples than another. Example: 990 non-fraud transactions and 10 fraud transactions.

**Why accuracy can mislead:** a model that predicts “not fraud” for every transaction gets 99% accuracy in that example but detects zero fraud cases. Evaluate precision, recall, F1-score, confusion matrix, and PR-AUC (especially for rare positive classes); choose metrics based on the business cost of false positives and false negatives.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight

X_imb, y_imb = make_classification(
    n_samples=2000, n_features=12, n_informative=5, n_redundant=2,
    weights=[0.95, 0.05], flip_y=0.01, random_state=42
)
Xtr, Xte, ytr, yte = train_test_split(
    X_imb, y_imb, test_size=0.25, random_state=42, stratify=y_imb
)
pd.Series(ytr).value_counts(normalize=True).rename("training proportion")

In [ ]:
# Baseline that predicts only the majority class
majority_prediction = np.zeros_like(yte)
print("Accuracy:", round(accuracy_score(yte, majority_prediction), 3))
print("Precision:", round(precision_score(yte, majority_prediction, zero_division=0), 3))
print("Recall:", round(recall_score(yte, majority_prediction, zero_division=0), 3))
print("F1:", round(f1_score(yte, majority_prediction, zero_division=0), 3))
print(confusion_matrix(yte, majority_prediction))

In [ ]:
# Compare ordinary logistic regression with class-weighted logistic regression
models = {
    "Default": LogisticRegression(max_iter=2000),
    "Class weighted": LogisticRegression(max_iter=2000, class_weight="balanced")
}
rows = []
for name, model in models.items():
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    rows.append({
        "model": name,
        "accuracy": accuracy_score(yte, pred),
        "precision": precision_score(yte, pred, zero_division=0),
        "recall": recall_score(yte, pred, zero_division=0),
        "f1": f1_score(yte, pred, zero_division=0)
    })
pd.DataFrame(rows).set_index("model").round(3)

## 2. Ways to handle imbalance

- **Random oversampling:** duplicates minority-class training examples. Simple, but can overfit repeated records.
- **Random undersampling:** removes majority-class examples. Faster, but may discard useful information.
- **SMOTE:** creates synthetic minority examples by interpolating between minority neighbors. Use only on training folds; it can create unrealistic examples for some data.
- **Borderline-SMOTE:** focuses synthetic generation near class boundaries; may help in some cases but can also amplify noise.
- **Class weights:** penalize mistakes on minority examples more heavily without generating or removing rows. Often a good baseline.

| Technique | Consider when | Caution |
|---|---|---|
| Random oversampling | Small dataset, want to preserve majority examples | Overfitting risk |
| Random undersampling | Very large majority class, speed matters | Information loss |
| SMOTE | Numeric features and minority examples have meaningful neighborhoods | Must not resample before splitting; categorical data needs suitable methods |
| Borderline-SMOTE | Boundary cases are especially important | Noisy boundaries can be amplified |
| Class weights | Want a simple baseline without changing sample count | Tune weights/threshold and validate |

Do not rebalance validation or test sets: keep them representative of the real deployment distribution. If using SMOTE, install `imbalanced-learn` and put its sampler in an `imblearn.pipeline.Pipeline`, so resampling occurs only during training. Example code is provided below as an optional cell.

In [ ]:
# OPTIONAL: run only if imbalanced-learn is installed.
# !pip install imbalanced-learn
#
# from imblearn.over_sampling import SMOTE, BorderlineSMOTE, RandomOverSampler
# from imblearn.under_sampling import RandomUnderSampler
# from imblearn.pipeline import Pipeline as ImbPipeline
# from sklearn.preprocessing import StandardScaler
#
# smote_model = ImbPipeline([
#     ("scale", StandardScaler()),
#     ("smote", SMOTE(random_state=42)),
#     ("model", LogisticRegression(max_iter=2000))
# ])
# smote_model.fit(Xtr, ytr)  # sampler acts only during fit on training data
# smote_pred = smote_model.predict(Xte)
# print(classification_report(yte, smote_pred, zero_division=0))

---
# Notebook 12 – Train, Validation & Test Data
## 1. Why split data?

- **Training set:** fits model parameters and preprocessing transformations.
- **Validation set:** compares models, tunes hyperparameters, and selects decision thresholds.
- **Test set:** final, one-time estimate of performance on unseen data.

`random_state` makes randomized splits reproducible. `stratify=y` helps preserve class proportions for classification. For time-dependent data, use a chronological split instead of a random split.

In [ ]:
from sklearn.model_selection import train_test_split

X_demo, y_demo = make_classification(
    n_samples=1000, n_features=8, n_informative=4,
    weights=[0.8, 0.2], random_state=42
)

# 60% train, 20% validation, 20% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X_demo, y_demo, test_size=0.40, random_state=42, stratify=y_demo
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)
print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)
print("Positive rate:", {
    "train": round(y_train.mean(), 3),
    "validation": round(y_val.mean(), 3),
    "test": round(y_test.mean(), 3)
})

In [ ]:
# INCORRECT: fit preprocessing on all data before splitting (leakage).
from sklearn.preprocessing import StandardScaler

scaler_wrong = StandardScaler()
X_scaled_all_wrong = scaler_wrong.fit_transform(X_demo)
Xtr_wrong, Xte_wrong, ytr_wrong, yte_wrong = train_test_split(
    X_scaled_all_wrong, y_demo, test_size=0.2, random_state=42, stratify=y_demo
)
print("Incorrect workflow: scaler saw the test set before split.")

In [ ]:
# CORRECT: split first, fit preprocessing only on training data.
scaler_correct = StandardScaler()
Xtr_correct = scaler_correct.fit_transform(X_train)
Xval_correct = scaler_correct.transform(X_val)
Xtest_correct = scaler_correct.transform(X_test)
print("Correct workflow shapes:", Xtr_correct.shape, Xval_correct.shape, Xtest_correct.shape)

### Temporal splitting

For forecasting or any problem where future data must not influence the past, sort by time and split chronologically. Do not randomly shuffle time-dependent records unless the problem setup genuinely allows it.

In [ ]:
dates = pd.date_range("2024-01-01", periods=12, freq="MS")
time_df = pd.DataFrame({"date": dates, "sales": np.arange(12) * 10 + 100})
train_time = time_df[time_df["date"] < "2024-09-01"]
test_time = time_df[time_df["date"] >= "2024-09-01"]
print("Train dates:", train_time["date"].min().date(), "to", train_time["date"].max().date())
print("Test dates:", test_time["date"].min().date(), "to", test_time["date"].max().date())

---
# Notebook 13 – Preventing Data Leakage
## 1. What is data leakage?

Data leakage occurs when training or model-selection steps use information that would not genuinely be available at prediction time. It often creates impressive validation scores that fail in real deployment.

Types:
- **Target leakage:** a feature directly or indirectly contains the answer.
- **Train-test contamination:** information from test records influences training.
- **Feature leakage:** a feature is created using information unavailable at prediction time.
- **Preprocessing leakage:** imputers, scalers, feature selectors, or resampling are fit before splitting.
- **Temporal leakage:** future information is used to predict the past.

In [ ]:
# Example: a "final_status" field is only known after an outcome.
leak_df = pd.DataFrame({
    "hours_studied": [1, 2, 3, 4, 5, 6, 7, 8],
    "attendance":    [50, 55, 60, 65, 70, 75, 80, 90],
    "passed_exam":   [0, 0, 0, 0, 1, 1, 1, 1]
})
# A fabricated field that directly reveals the target:
leak_df["final_status"] = leak_df["passed_exam"].map({0: "failed", 1: "passed"})
leak_df

### Incorrect workflow → Correct workflow

**Incorrect:** include `final_status` as an input feature when predicting `passed_exam`. It is only known after the exam and directly reveals the answer.

**Correct:** remove it from predictors and use only information available at prediction time. Similarly, split first and fit all preprocessing/selection only on the training portion.

In [ ]:
# INCORRECT: target-leaking feature
X_leaky = leak_df[["hours_studied", "attendance", "final_status"]]
y_leaky = leak_df["passed_exam"]
print("Do not train with final_status; it directly encodes the outcome.")

In [ ]:
# CORRECT: only pre-outcome features
X_clean = leak_df[["hours_studied", "attendance"]]
y_clean = leak_df["passed_exam"]
X_train_clean, X_test_clean, y_train_clean, y_test_clean = train_test_split(
    X_clean, y_clean, test_size=0.25, random_state=42, stratify=y_clean
)
print(X_train_clean.columns.tolist())

In [ ]:
# INCORRECT preprocessing leakage: impute/scale before splitting.
from sklearn.impute import SimpleImputer
leak_X = pd.DataFrame({
    "income": [30, 35, np.nan, 45, 50, 1000, 55, 60],
    "age": [20, 25, 30, np.nan, 40, 45, 50, 55]
})
leak_y = np.array([0, 0, 0, 0, 1, 1, 1, 1])

# This is intentionally shown as an anti-pattern:
imputer_wrong = SimpleImputer(strategy="median")
X_imputed_all_wrong = imputer_wrong.fit_transform(leak_X)
Xtr_bad, Xte_bad, ytr_bad, yte_bad = train_test_split(
    X_imputed_all_wrong, leak_y, test_size=0.25, random_state=42, stratify=leak_y
)
print("Incorrect: medians were calculated using all rows.")

In [ ]:
# CORRECT: split raw data first, then fit imputer/scaler on training only.
Xtr_raw, Xte_raw, ytr, yte = train_test_split(
    leak_X, leak_y, test_size=0.25, random_state=42, stratify=leak_y
)
from sklearn.pipeline import Pipeline
safe_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
Xtr_safe = safe_preprocessor.fit_transform(Xtr_raw)
Xte_safe = safe_preprocessor.transform(Xte_raw)
print("Training and test shapes:", Xtr_safe.shape, Xte_safe.shape)

### Leakage prevention checklist

1. Define the prediction moment: what is actually known then?
2. Remove post-outcome fields and target-derived columns.
3. Split before fitting imputers, scalers, encoders, feature selectors, or resamplers.
4. Use pipelines so each cross-validation fold fits its own preprocessing.
5. For time series, use chronological splits and lag features that only use past values.
6. Keep the test set untouched until the final evaluation.
7. Check suspiciously perfect metrics, duplicate entities across splits, and features that encode the label.

---
# Notebook 14 – Scikit-learn Preprocessing Pipeline
## 1. Why use a pipeline?

A pipeline bundles preprocessing and a model into one reproducible object. It reduces accidental leakage, keeps training and inference transformations consistent, and makes cross-validation safer.

- `Pipeline`: chains steps in order.
- `ColumnTransformer`: applies different transformations to different columns.
- `fit()`: learns parameters from data.
- `transform()`: applies learned transformations.
- `fit_transform()`: fits and transforms in one call; commonly used on training data.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Small mixed-type example with missing values
rng = np.random.default_rng(21)
n = 500
customer_df = pd.DataFrame({
    "age": rng.integers(18, 75, n).astype(float),
    "income": rng.normal(55000, 18000, n).clip(10000, 150000),
    "city": rng.choice(["Chennai", "Salem", "Coimbatore"], n),
    "plan": rng.choice(["basic", "plus", "premium"], n),
})
prob = 1 / (1 + np.exp(-(
    -2.0 + 0.025 * (customer_df["age"] - 40)
    + 0.000015 * (customer_df["income"] - 50000)
    + (customer_df["plan"] == "premium") * 0.9
)))
customer_df["churn"] = rng.binomial(1, prob)

# Add missing values to demonstrate imputation
customer_df.loc[rng.choice(n, 25, replace=False), "age"] = np.nan
customer_df.loc[rng.choice(n, 20, replace=False), "city"] = np.nan
customer_df.head()

In [ ]:
X = customer_df.drop(columns="churn")
y = customer_df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_features = ["age", "income"]
categorical_features = ["city", "plan"]

# Numerical features: impute missing values -> scale
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical features: impute missing values -> one-hot encode
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Apply the correct pipeline to each column group
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# Complete model pipeline: preprocessing -> classifier
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000))
])

model_pipeline.fit(X_train, y_train)
predictions = model_pipeline.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, predictions), 3))
print(classification_report(y_test, predictions, zero_division=0))

### Explain every stage

1. **Numerical features:** `SimpleImputer(strategy="median")` fills missing numeric values using medians learned from training data.
2. **Scaling:** `StandardScaler` standardizes numeric columns using training-set statistics.
3. **Categorical features:** `SimpleImputer(strategy="most_frequent")` fills missing categories using the most common training category.
4. **Encoding:** `OneHotEncoder(handle_unknown="ignore")` converts categories into numeric indicator columns and safely handles unseen categories at prediction time.
5. **ColumnTransformer:** applies the numerical and categorical pipelines to the specified column groups and combines the results.
6. **Pipeline:** fits preprocessing and the classifier together. Calling `model_pipeline.predict(new_data)` automatically applies the same learned transformations.

### `fit()`, `transform()`, and `fit_transform()`

- `fit(X_train, y_train)`: learns imputation values, scaling statistics, encoding categories, and model parameters.
- `transform(X)`: applies already-learned preprocessing to new data.
- `fit_transform(X_train)`: learns preprocessing and transforms the training data in one call.
- Never call `fit_transform()` on the test data. Use `transform()` through the fitted pipeline instead.

The pipeline keeps preprocessing consistent and helps prevent leakage. It does not automatically solve every leakage risk: you must still choose valid features, split by time/entity when required, and ensure each feature is available at prediction time.